# How to connect a chat bot to your memory service

In [16]:
import dotenv

dotenv.load_dotenv("../../.env", override=True)


True

In [17]:
from langgraph_sdk import get_client

# Update to your URL. Copy this from page of your LangGraph Deployment
deployment_url = "http://127.0.0.1:2024"

client = get_client(url=deployment_url)

## Example Chat Bot

The bot fetches user memories my semantic similarity, templates them, then responds!

In [114]:
import os
import uuid
from datetime import datetime, timezone
from typing import List, Optional

import langsmith
from langchain.chat_models import init_chat_model
from langchain_core.messages import AnyMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, StateGraph, add_messages
from langgraph_sdk import get_client
from pydantic import BaseModel, Field
from typing_extensions import Annotated, TypedDict

from memory_agent import (
    constants,
    settings,
    utils,
)


class ChatState(TypedDict):
    """The state of the chatbot."""

    messages: Annotated[List[AnyMessage], add_messages]
    user_memories: List[dict]


class ChatConfigurable(TypedDict):
    """The configurable fields for the chatbot."""

    user_id: str
    thread_id: str
    memory_service_url: str = ""
    model: str
    delay: Optional[float]


def _ensure_configurable(config: RunnableConfig) -> ChatConfigurable:
    """Ensure the configuration is valid."""
    return ChatConfigurable(
        user_id=config["configurable"]["user_id"],
        thread_id=config["configurable"]["thread_id"],
        mem_assistant_id=config["configurable"]["mem_assistant_id"],
        memory_service_url=config["configurable"].get(
            "memory_service_url", os.environ.get("MEMORY_SERVICE_URL", "")
        ),
        model=config["configurable"].get(
            # "model", "accounts/fireworks/models/firefunction-v2"
            "model", "gpt-4o-mini"
        ),
        # delay=config["configurable"].get("delay", 60),
        delay=config["configurable"].get("delay", 1),
    )


PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful and friendly chatbot. Get to know the user!"
            " Ask questions! Be spontaneous!"
            "{user_info}\n\nSystem Time: {time}",
        ),
        ("placeholder", "{messages}"),
    ]
).partial(
    time=lambda: datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
)


@langsmith.traceable
def format_query(messages: List[AnyMessage]) -> str:
    """Format the query for the user's memories."""
    # This is quite naive :)
    return " ".join([str(m.content) for m in messages if m.type == "human"][-5:])


async def query_memories(state: ChatState, config: RunnableConfig) -> ChatState:
    """Query the user's memories."""
    print("query_memories")
    configurable: ChatConfigurable = config["configurable"]
    user_id = configurable["user_id"]
    index = utils.get_index()
    embeddings = utils.get_embeddings()

    query = format_query(state["messages"])
    vec = await embeddings.aembed_query(query)
    # You can also filter by memory type, etc. here.
    with langsmith.trace(
        "pinecone_query", inputs={"query": query, "user_id": user_id}
    ) as rt:
        response = index.query(
            vector=vec,
            filter={"user_id": {"$eq": str(user_id)}},
            include_metadata=True,
            top_k=10,
            namespace=settings.SETTINGS.pinecone_namespace,
        )
        rt.outputs["response"] = response
    memories = []
    if matches := response.get("matches"):
        memories = [m["metadata"][constants.PAYLOAD_KEY] for m in matches]
    print(f"memories: {memories}")
    return {
        "user_memories": memories,
    }


@langsmith.traceable
def format_memories(memories: List[dict]) -> str:
    """Format the user's memories."""
    if not memories:
        return ""
    # Note Bene: You can format better than this....
    memories = "\n".join(str(m) for m in memories)
    return f"""

## Memories

You have noted the following memorable events from previous interactions with the user.
<memories>
{memories}
</memories>
"""


async def bot(state: ChatState, config: RunnableConfig) -> ChatState:
    """Prompt the bot to resopnd to the user, incorporating memories (if provided)."""
    print("bot")
    configurable = _ensure_configurable(config)
    model = init_chat_model(configurable["model"])
    chain = PROMPT | model
    memories = format_memories(state["user_memories"])
    m = await chain.ainvoke(
        {
            "messages": state["messages"],
            "user_info": memories,
        },
        config,
    )

    return {
        "messages": [m],
    }


class MemorableEvent(BaseModel):
    """A memorable event."""

    description: str
    participants: List[str] = Field(
        description="Names of participants in the event and their relationship to the user."
    )


async def post_messages(state: ChatState, config: RunnableConfig) -> ChatState:
    print("post_messages")
    """Query the user's memories."""
    configurable = _ensure_configurable(config)
    langgraph_client = get_client(url=configurable["memory_service_url"])
    thread_id = config["configurable"]["thread_id"]
    # Hash "memory_{thread_id}" to get a new uuid5 for the memory id
    memory_thread_id = uuid.uuid5(uuid.NAMESPACE_URL, f"memory_{thread_id}")
    try:
        await langgraph_client.threads.get(thread_id=memory_thread_id)
    except Exception:
        await langgraph_client.threads.create(thread_id=memory_thread_id)

    await langgraph_client.runs.create(
        memory_thread_id,
        assistant_id=configurable["mem_assistant_id"],
        input={
            "messages": state["messages"],  # the service dedupes messages
        },
        config={
            "configurable": {
                "user_id": configurable["user_id"],
            },
        },
        # multitask_strategy="rollback",
        multitask_strategy="enqueue",
    )

    return {
        "messages": [],
    }


builder = StateGraph(ChatState, ChatConfigurable)
builder.add_node(query_memories)
builder.add_node(bot)
builder.add_node(post_messages)
builder.add_edge(START, "query_memories")
builder.add_edge("query_memories", "bot")
builder.add_edge("bot", "post_messages")

chat_graph = builder.compile(checkpointer=MemorySaver())

In [94]:
result = await client.assistants.search()
result

[{'assistant_id': 'eb11b265-df51-40a3-a345-5bfc6b61ab24',
  'graph_id': 'memory',
  'config': {'configurable': {'delay': 4,
    'schemas': {'MemorableEvent': {'system_prompt': "Extract any memorable events from the user's messages that you would like to remember.",
      'update_mode': 'insert',
      'function': {'description': 'A memorable event.',
       'properties': {'description': {'title': 'Description',
         'type': 'string'},
        'participants': {'description': 'Names of participants in the event and their relationship to the user.',
         'items': {'type': 'string'},
         'title': 'Participants',
         'type': 'array'}},
       'required': ['description', 'participants'],
       'title': 'MemorableEvent',
       'type': 'object'}}}}},
  'metadata': {},
  'name': 'Untitled',
  'created_at': '2025-02-21T13:37:36.248564+00:00',
  'updated_at': '2025-02-21T13:37:36.248564+00:00',
  'version': 1},
 {'assistant_id': 'a86c8600-ddcb-4f1c-ae39-b860c01c3dc0',
  'graph

In [95]:
mem_assistant = await client.assistants.create(
    graph_id="memory",
    config={
        "configurable": {
            "delay": 4,  # seconds wait before considering a thread as "completed"
            "schemas": {
                "MemorableEvent": {
                    "system_prompt": "Extract any memorable events from the user's"
                    " messages that you would like to remember.",
                    "update_mode": "insert",
                    "function": MemorableEvent.schema(),
                },
            },
        }
    },
)

/var/folders/q_/nkt84m3j2nb1ppf5cn3v6zq00000gn/T/ipykernel_92158/1171363277.py:11: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  "function": MemorableEvent.schema(),


In [96]:
mem_assistant = (await client.assistants.search(graph_id="memory"))[0]
mem_assistant

{'assistant_id': '9d5a371c-abd2-4dd4-a1dd-21fb98dad205',
 'graph_id': 'memory',
 'config': {'configurable': {'delay': 4,
   'schemas': {'MemorableEvent': {'system_prompt': "Extract any memorable events from the user's messages that you would like to remember.",
     'update_mode': 'insert',
     'function': {'description': 'A memorable event.',
      'properties': {'description': {'title': 'Description', 'type': 'string'},
       'participants': {'description': 'Names of participants in the event and their relationship to the user.',
        'items': {'type': 'string'},
        'title': 'Participants',
        'type': 'array'}},
      'required': ['description', 'participants'],
      'title': 'MemorableEvent',
      'type': 'object'}}}}},
 'metadata': {},
 'name': 'Untitled',
 'created_at': '2025-02-21T13:40:19.799576+00:00',
 'updated_at': '2025-02-21T13:40:19.799576+00:00',
 'version': 1}

In [97]:
import uuid

user_id = str(uuid.uuid4())  # more permanent

In [98]:
thread_id = str(uuid.uuid4())  # can adjust
result = await client.threads.create(thread_id=thread_id)
result

{'thread_id': 'f2020bda-37a6-4f8a-a9b4-510ed8a9b4f4',
 'created_at': '2025-02-21T13:40:25.224654+00:00',
 'updated_at': '2025-02-21T13:40:25.224658+00:00',
 'metadata': {},
 'status': 'idle',
 'config': {},
 'values': None}

In [116]:
class Chat:
    def __init__(self, user_id: str, thread_id: str):
        self.thread_id = thread_id
        self.user_id = user_id

    async def __call__(self, query: str) -> str:
        chunks = chat_graph.astream_events(
            input={
                "messages": [("user", query)],
            },
            config={
                "configurable": {
                    "user_id": self.user_id,
                    "thread_id": self.thread_id,
                    "memory_service_url": deployment_url,
                    "mem_assistant_id": mem_assistant["assistant_id"],
                    "delay": 4,
                }
            },
            version="v2",
        )
        res = ""
        async for event in chunks:
            if event.get("event") == "on_chat_model_stream":
                tok = event["data"]["chunk"].content
                print(tok, end="")
                res += tok
        return res

In [117]:
chat = Chat(user_id, thread_id)
chat

In [118]:
_ = await chat("Hi there")

query_memories
memories: ['{"description":"Planning a surprise party for friend Steve to cheer him up after a rough month.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for a friend named Steve to cheer him up after a rough month.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for a friend named Steve who has been having a rough month.","participants":["user","Steve"]}', '{"description":"User is planning a surprise party for their friend Steve to cheer him up after a rough month.","participants":["User","Steve"]}']
bot
Hi! How are you today? 😊 What’s on your mind?post_messages


In [119]:
_ = await chat(
    "I've been planning a surprise party for my friend steve. "
    "He has been having a rough month and I want it to be special."
)

query_memories
memories: ['{"description":"Planning a surprise party for a friend named Steve who has been having a rough month.","participants":["user","Steve"]}', '{"description":"Planning a surprise party for a friend named Steve to cheer him up after a rough month.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for friend Steve to cheer him up after a rough month.","participants":["User","Steve"]}', '{"description":"User is planning a surprise party for their friend Steve to cheer him up after a rough month.","participants":["User","Steve"]}']
bot
That’s so thoughtful of you! 🎉 What do you have in mind for the party? Any specific themes or ideas to make it special for Steve?post_messages


In [120]:
_ = await chat(
    "Steve really likes crocheting. Maybe I can do something with that? Or is that dumb... "
)

query_memories
memories: ['{"description":"Planning a surprise party for a friend named Steve who has been having a rough month.","participants":["user","Steve"]}', '{"description":"Planning a surprise party for a friend named Steve who has been having a rough month.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for a friend named Steve to cheer him up after a rough month.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for friend Steve to cheer him up after a rough month.","participants":["User","Steve"]}', '{"description":"User is planning a surprise party for their friend Steve to cheer him up after a rough month.","participants":["User","Steve"]}']
bot
Not dumb at all! That sounds like a fantastic idea! You could incorporate crocheting into the decorations, or even set up a little crocheting station for guests to join in. Maybe have a crochet-themed cake or party favors? What do you think?post_messages


In [121]:
_ = await chat("He's also into capoeira...")

query_memories
memories: ['{"description":"Planning a surprise party for a friend named Steve who has been having a rough month.","participants":["user","Steve"]}', '{"description":"Planning a surprise party for a friend named Steve who has been having a rough month.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for a friend named Steve who has been having a rough month.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for a friend named Steve to cheer him up after a rough month.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for friend Steve to cheer him up after a rough month.","participants":["User","Steve"]}', '{"description":"User is planning a surprise party for their friend Steve to cheer him up after a rough month.","participants":["User","Steve"]}']
bot
That’s awesome! You could combine his love for crocheting and capoeira in some creative ways. Maybe you could make crocheted capoeir

In [122]:
_ = await chat(
    "Oh that's a cool idea. One time i took classes from this studio nearby. Wonder if they have any recs. "
)

query_memories
memories: ['{"description":"Planning a surprise party for a friend named Steve who has been having a rough month.","participants":["user","Steve"]}', '{"description":"Steve enjoys crocheting, and the user is considering incorporating that theme into the surprise party.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for a friend named Steve who has been having a rough month.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for a friend named Steve who has been having a rough month.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for a friend named Steve to cheer him up after a rough month.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for friend Steve to cheer him up after a rough month.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for Steve who has been having a rough month.","participants":["User","Steve"]}', '{"

In [123]:
_ = await chat("Idk. Anyways - how are you doing?")

query_memories
memories: ['{"description":"Planning a surprise party for a friend who likes crocheting and capoeira.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for a friend named Steve who has been having a rough month.","participants":["user","Steve"]}', '{"description":"Planning a surprise party for a friend named Steve who has been having a rough month.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for a friend named Steve who has been having a rough month.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for a friend named Steve to cheer him up after a rough month.","participants":["User","Steve"]}', '{"description":"Steve enjoys crocheting, and the user is considering incorporating that theme into the surprise party.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for friend Steve to cheer him up after a rough month.","participants":["User","Steve"]}', 

In [124]:
_ = await chat("My name is Ken btw")

query_memories
memories: ['{"description":"Planning a surprise party for a friend who likes crocheting and capoeira.","participants":["User","Steve"]}', '{"description":"User is planning a surprise party for their friend Steve, who has been having a rough month. The party will incorporate Steve\'s interests in crocheting and capoeira.","participants":["User","Steve"]}', '{"description":"Steve enjoys crocheting, and the user is considering incorporating that theme into the surprise party.","participants":["User","Steve"]}', '{"description":"User took capoeira classes from a nearby studio.","participants":["User"]}', '{"description":"Planning a surprise party for a friend named Steve who has been having a rough month.","participants":["user","Steve"]}', '{"description":"Planning a surprise party for a friend named Steve to cheer him up after a rough month.","participants":["User","Steve"]}', '{"description":"User\'s friend, Steve, has been having a rough month.","participants":["User","S

## Convo 2

Our memory is configured only to consider a thread "ready to process" if has been inactive for a minute.
We'll wait for things to populate

In [49]:
import asyncio

await asyncio.sleep(60)

CancelledError: 

In [125]:
thread_id_2 = uuid.uuid4()

In [126]:
chat2 = Chat(user_id, thread_id_2)

In [127]:
_ = await chat2("Remember me?")

query_memories
memories: ['{"description":"Planning a surprise party for a friend named Steve to cheer him up after a rough month.","participants":["User","Steve"]}', '{"description":"User\'s friend, Steve, has been having a rough month.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for friend Steve to cheer him up after a rough month.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for a friend named Steve who has been having a rough month.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for a friend named Steve who has been having a rough month.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for a friend named Steve who has been having a rough month.","participants":["user","Steve"]}', '{"description":"User is planning a surprise party for their friend Steve to cheer him up after a rough month.","participants":["User","Steve"]}', '{"description":"Planning a s

In [128]:
_ = await chat2("wdy remember??")

query_memories
memories: ['{"description":"User\'s friend, Steve, has been having a rough month.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for a friend named Steve to cheer him up after a rough month.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for friend Steve to cheer him up after a rough month.","participants":["User","Steve"]}', '{"description":"Ken introduced himself in the conversation.","participants":["Ken"]}', '{"description":"Planning a surprise party for a friend named Steve who has been having a rough month.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for a friend named Steve who has been having a rough month.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for a friend named Steve who has been having a rough month.","participants":["user","Steve"]}', '{"description":"User is planning a surprise party for their friend Steve to cheer him u

In [129]:
_ = await chat2("Oh planning is going alright!")

query_memories
memories: ['{"description":"Planning a surprise party for a friend named Steve who has been having a rough month.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for a friend named Steve who has been having a rough month.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for a friend named Steve who has been having a rough month.","participants":["user","Steve"]}', '{"description":"Planning a surprise party for friend Steve to cheer him up after a rough month.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for a friend named Steve to cheer him up after a rough month.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for Steve who has been having a rough month.","participants":["User","Steve"]}', '{"description":"Planning a surprise party for Steve to cheer him up after a tough month.","participants":["User","Steve"]}', '{"description":"User is planning